In [22]:
import sys
print(sys.version)

3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]


In [23]:
import os
import pdfplumber
import tiktoken
from dotenv import load_dotenv
from openai import OpenAI

In [24]:
load_dotenv()

True

In [25]:
client = OpenAI(api_key = os.getenv("OPEN_API_KEY"))

In [26]:
print(client)

In [27]:
PDF_FOLDER = "pdfs"
OUTPUT_FOLDER = "outputs"

In [28]:
# ------------------------------
# Chunk long text safely
# ------------------------------
def chunk_text(text, max_tokens=12000):
    enc = tiktoken.encoding_for_model("gpt-4o-mini")
    tokens = enc.encode(text)

    chunks = []
    for i in range(0, len(tokens), max_tokens):
        chunk = enc.decode(tokens[i:i + max_tokens])
        chunks.append(chunk)

    return chunks

In [29]:
# ------------------------------
# Extract text from PDF
# ------------------------------
def extract_text_from_pdf(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text() or ""
            text += page_text + "\n"
    return text

In [30]:
# ------------------------------
# Generate MCQs for a chunk
# ------------------------------
def generate_mcqs_from_chunk(chunk_text):
    prompt = f"""
You are an expert textbook instructor.

Your task: Based on the following textbook content, generate:

1. A list of topics covered  
2. For each topic:  
   - 5 MCQs (4 options each)  
   - The correct answers clearly stated

Ensure clarity, structure, and accuracy.

TEXT CHUNK:
{chunk_text}
"""

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
    )

    return response.choices[0].message.content

In [31]:
# ------------------------------
# Main processing
# ------------------------------
def process_pdfs():
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)

    pdf_files = [f for f in os.listdir(PDF_FOLDER) if f.endswith(".pdf")]
    if not pdf_files:
        print("❌ No PDFs found in 'pdfs/' folder.")
        return

    for pdf in pdf_files:
        print(f"\n📘 Processing: {pdf}")
        pdf_path = os.path.join(PDF_FOLDER, pdf)

        text = extract_text_from_pdf(pdf_path)
        chunks = chunk_text(text)

        print(f"➡ Extracted text length: {len(text)} characters")
        print(f"➡ Number of chunks: {len(chunks)}")

        output_file = os.path.join(OUTPUT_FOLDER, f"{pdf}_MCQs.md")
        with open(output_file, "w", encoding="utf-8") as f:
            f.write(f"# MCQs for {pdf}\n\n")

            for i, chunk in enumerate(chunks):
                print(f"   🧩 Processing chunk {i+1}/{len(chunks)}...")
                mcq_output = generate_mcqs_from_chunk(chunk)

                f.write(f"## Chunk {i+1}\n")
                f.write(mcq_output + "\n\n")

        print(f"✅ Completed → Saved to: {output_file}")

In [32]:
process_pdfs()


📘 Processing: Electrical_MEMU Troubleshooting Manual.pdf
➡ Extracted text length: 214972 characters
➡ Number of chunks: 6
   🧩 Processing chunk 1/6...
   🧩 Processing chunk 2/6...
   🧩 Processing chunk 3/6...
   🧩 Processing chunk 4/6...
   🧩 Processing chunk 5/6...
   🧩 Processing chunk 6/6...
✅ Completed → Saved to: outputs\Electrical_MEMU Troubleshooting Manual.pdf_MCQs.md

📘 Processing: EMU MEMU TECHNOLOGY IN IR.docx_.pdf
➡ Extracted text length: 264042 characters
➡ Number of chunks: 6
   🧩 Processing chunk 1/6...
   🧩 Processing chunk 2/6...
   🧩 Processing chunk 3/6...
   🧩 Processing chunk 4/6...
   🧩 Processing chunk 5/6...
   🧩 Processing chunk 6/6...
✅ Completed → Saved to: outputs\EMU MEMU TECHNOLOGY IN IR.docx_.pdf_MCQs.md

📘 Processing: Reading material EMU-Electrical.pdf
➡ Extracted text length: 64872 characters
➡ Number of chunks: 2
   🧩 Processing chunk 1/2...
   🧩 Processing chunk 2/2...
✅ Completed → Saved to: outputs\Reading material EMU-Electrical.pdf_MCQs.md

📘 Pr

Cannot set gray non-stroke color because /'R10' is an invalid float value


✅ Completed → Saved to: outputs\Revised Final draft Specification RDSO_PE_SPEC_EMU_0163(Rev_ 3)_2021 with annexures final for upload.pdf_MCQs.md

📘 Processing: Troubleshooting directory for 25 kV AC EMU-MEMU.pdf
➡ Extracted text length: 71272 characters
➡ Number of chunks: 2
   🧩 Processing chunk 1/2...
   🧩 Processing chunk 2/2...
✅ Completed → Saved to: outputs\Troubleshooting directory for 25 kV AC EMU-MEMU.pdf_MCQs.md
